# 11 Improved Features

**Purpose.** Feature-engineering variants on the profile / map representations.

**Thesis reference.** §5 Representation Comparison

**Inputs / outputs.** See the data-layout table in the repository README. Paths near the top of the notebook (e.g. `PROFILE_DIR`, `IMAGE_ROOT`, `LABELS`) are relative to the repository root and assume the folder layout described there.

---

## Configuration

In [ ]:
PROFILE_DIR  = "profiles"
LABELS_FILE  = "labels/labels_main_any_ckd.json"   # or labels_id_matched.json
OUT_FILE     = "improved_results.json"
NORM         = "pqn"      # "pqn" or "tic"
MAX_FEATURES = 4000       # variance cap, fitted per training fold
CV_REPEATS   = 5
PERMUTATIONS = 5
ELASTICNET   = True


## Code

## Run

In [5]:
import argparse
import json
import warnings
from pathlib import Path

import numpy as np
from sklearn.cross_decomposition import PLSRegression
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import RepeatedStratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")
RANDOM_STATE = 42

# Normalisation
def tic_normalise(X):
    t = X.sum(axis=1, keepdims=True)
    t[t == 0] = 1.0
    return X / t * 1e6

def pqn_normalise(X, ref=None):
    """
    Probabilistic quotient normalisation.

    Each spectrum is divided by the median of its quotients against a reference
    spectrum, computed over features present in both. This resists distortion
    by a few very intense species, which plain TIC normalisation does not.
    """
    Xt = tic_normalise(X)
    if ref is None:
        ref = np.median(Xt, axis=0)
    out = np.empty_like(Xt)
    for i in range(len(Xt)):
        mask = (ref > 0) & (Xt[i] > 0)
        q = np.median(Xt[i, mask] / ref[mask]) if mask.sum() > 10 else 1.0
        out[i] = Xt[i] / (q if q > 0 else 1.0)
    return out, ref

def log_transform(X):
    return np.log1p(X)

# Filtering (fitted on training data only)
class FeatureFilter(BaseEstimator):
    """Drop features that are rarely present or carry negligible variance."""

    def __init__(self, min_presence=0.5, var_quantile=0.25, max_features=4000):
        self.min_presence = min_presence
        self.var_quantile = var_quantile
        self.max_features = max_features

    def fit(self, X, y=None):
        presence = (X > 0).mean(axis=0)
        keep = presence >= self.min_presence
        if keep.sum() < 50:
            keep = presence >= np.percentile(presence, 50)
        v = X[:, keep].var(axis=0)
        thr = np.quantile(v, self.var_quantile)
        idx = np.where(keep)[0]
        self.keep_ = idx[v > thr]
        if len(self.keep_) < 20:
            self.keep_ = idx
        # Cap dimensionality by variance. This is unsupervised --- it never
        # looks at y --- so it cannot leak class information, and it keeps
        # the regularised solvers tractable on tens of thousands of features.
        if self.max_features and len(self.keep_) > self.max_features:
            vk = X[:, self.keep_].var(axis=0)
            self.keep_ = self.keep_[np.argsort(vk)[::-1][:self.max_features]]
            self.keep_.sort()
        return self

    def transform(self, X):
        return X[:, self.keep_]

# PLS-DA
class PLSDA(BaseEstimator, ClassifierMixin):
    """Partial least squares discriminant analysis, the metabolomics standard."""

    def __init__(self, n_components=5):
        self.n_components = n_components

    def fit(self, X, y):
        self.pls_ = PLSRegression(
            n_components=min(self.n_components, X.shape[1], len(X) - 1))
        self.pls_.fit(X, np.asarray(y, dtype=float))
        self.classes_ = np.array([0, 1])
        return self

    def predict_proba(self, X):
        s = self.pls_.predict(X).ravel()
        p = 1.0 / (1.0 + np.exp(-(s - 0.5) * 6))
        return np.column_stack([1 - p, p])

    def predict(self, X):
        return (self.predict_proba(X)[:, 1] > 0.5).astype(int)

# Pipelines
def make_pipeline(model, min_presence=0.5, var_quantile=0.25,
                  max_features=4000, **kw):
    if model == "plsda":
        clf = PLSDA(n_components=kw.get("n_components", 5))
    elif model == "elasticnet":
        clf = LogisticRegression(penalty="elasticnet", solver="saga",
                                 l1_ratio=kw.get("l1_ratio", 0.5),
                                 C=kw.get("C", 0.1), max_iter=500,
                                 tol=1e-3, class_weight="balanced")
    else:
        clf = LogisticRegression(C=kw.get("C", 0.1), max_iter=3000,
                                 class_weight="balanced")
    return Pipeline([
        ("filter", FeatureFilter(min_presence, var_quantile, max_features)),
        ("scale", StandardScaler()),
        ("clf", clf),
    ])

def cv_auc(pipe, X, y, n_splits=5, n_repeats=5, seed=RANDOM_STATE):
    rskf = RepeatedStratifiedKFold(n_splits=n_splits, n_repeats=n_repeats,
                                   random_state=seed)
    aucs = []
    for tr, te in rskf.split(X, y):
        try:
            p = pipe.fit(X[tr], y[tr]).predict_proba(X[te])[:, 1]
            if len(np.unique(y[te])) > 1:
                aucs.append(roc_auc_score(y[te], p))
        except Exception:
            continue
    return (float(np.mean(aucs)), float(np.std(aucs))) if aucs else (0.5, 0.0)

# Feature sets
def build_feature_sets(prof_dir, pids_keep, norm):
    prof = Path(prof_dir)
    with open(prof / "profile_pids.json") as fh:
        all_pids = json.load(fh)
    idx = [all_pids.index(p) for p in pids_keep]

    sets = {}

    F = np.load(prof / "mz_profiles.npy")[idx]
    sets["profile_1d"] = F

    cm_path = prof / "coarse_maps.npy"
    if cm_path.exists():
        C = np.load(cm_path)[idx]                    # (n, RT, mz)
        sets["map_2d_flat"] = C.reshape(len(C), -1)
        # marginals: how intensity is distributed over elution time
        sets["rt_profile"] = C.sum(axis=2)
        # 2-D but coarsened on the RT axis, to limit dimensionality
        if C.shape[1] >= 8:
            k = C.shape[1] // 8
            sets["map_2d_rt8"] = C[:, :k * 8].reshape(
                len(C), 8, k, C.shape[2]).sum(axis=2).reshape(len(C), -1)

    out = {}
    for name, X in sets.items():
        X = X.astype(np.float64)
        Xn = pqn_normalise(X)[0] if norm == "pqn" else tic_normalise(X)
        out[name] = log_transform(Xn)
    return out

def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--profiles", required=True)
    ap.add_argument("--labels", required=True)
    ap.add_argument("--out", default="improved_results.json")
    ap.add_argument("--norm", choices=["tic", "pqn"], default="pqn")
    ap.add_argument("--n-repeats", type=int, default=5)
    ap.add_argument("--permutations", type=int, default=5)
    ap.add_argument("--max-features", type=int, default=4000,
                    help="variance cap, applied inside each training fold")
    ap.add_argument("--elasticnet", action="store_true",
                    help="also fit elastic net (slow on wide feature sets)")
    args = ap.parse_args()

    prof = Path(args.profiles)
    with open(prof / "profile_pids.json") as fh:
        all_pids = json.load(fh)
    with open(args.labels) as fh:
        labels = json.load(fh).get("labels")

    pids = [p for p in all_pids if p in labels]
    y = np.array([labels[p] for p in pids])
    print(f"Patients: {len(y)}  ({int(y.sum())} pos / {int((1-y).sum())} neg)")
    print(f"Normalisation: {args.norm.upper()}\n")

    feats = build_feature_sets(args.profiles, pids, args.norm)
    for k, v in feats.items():
        print(f"  {k:16s} {v.shape}")
    print()

    idx = np.arange(len(y))
    i_dev, i_test = train_test_split(idx, test_size=0.2, stratify=y,
                                     random_state=RANDOM_STATE)
    print(f"dev {len(i_dev)} | test {len(i_test)}\n")

    configs = [
        ("plsda_3",     dict(model="plsda", n_components=3)),
        ("plsda_5",     dict(model="plsda", n_components=5)),
        ("plsda_10",    dict(model="plsda", n_components=10)),
        ("ridge_C0.01", dict(model="ridge", C=0.01)),
        ("ridge_C0.1",  dict(model="ridge", C=0.1)),
    ]
    if args.elasticnet:
        configs += [("enet_C0.01", dict(model="elasticnet", C=0.01)),
                    ("enet_C0.1",  dict(model="elasticnet", C=0.1))]

    results, best = {}, None
    print(f"{'features':16s} {'model':13s} {'CV AUC':>16} {'TEST':>8}")
    print("-" * 60)
    for fname, X in feats.items():
        Xd, Xt = X[i_dev], X[i_test]
        yd, yt = y[i_dev], y[i_test]
        for cname, kw in configs:
            pipe = make_pipeline(max_features=args.max_features, **kw)
            m, s = cv_auc(pipe, Xd, yd, n_repeats=args.n_repeats)
            try:
                pipe.fit(Xd, yd)
                t = roc_auc_score(yt, pipe.predict_proba(Xt)[:, 1])
            except Exception:
                t = float("nan")
            key = f"{fname}__{cname}"
            results[key] = {"features": fname, "model": cname,
                            "cv_auc_mean": m, "cv_auc_std": s,
                            "test_auc": float(t)}
            print(f"{fname:16s} {cname:13s} {m:.4f} +/- {s:.4f} {t:8.4f}")
            if best is None or m > best[0]:
                best = (m, fname, cname, kw)
        print()

    # permutation control on the best configuration
    m, fname, cname, kw = best
    print(f"Best: {fname} / {cname}  CV {m:.4f}")
    print(f"\nPermutation control on that configuration "
          f"({args.permutations} shuffles)")
    rng = np.random.default_rng(RANDOM_STATE)
    Xd, yd = feats[fname][i_dev], y[i_dev]
    perms = []
    for r in range(args.permutations):
        yp = rng.permutation(yd)
        pm, _ = cv_auc(make_pipeline(max_features=args.max_features, **kw),
                       Xd, yp, n_repeats=1, seed=r)
        perms.append(pm)
        print(f"  run {r+1}: {pm:.4f}")
    pmean = float(np.mean(perms))
    print(f"  mean {pmean:.4f}")

    gap = m - pmean
    print(f"\n  observed {m:.4f} - permuted {pmean:.4f} = {gap:+.4f}")
    if gap < 0.03:
        print("  -> NO signal above the null for this configuration.")
    elif gap < 0.08:
        print("  -> Marginal. Do not rely on it without independent support.")
    else:
        print("  -> Signal above the null. Confirm on the ID-matched cohort.")

    results["_best"] = {"features": fname, "model": cname,
                        "cv_auc": m, "permuted_auc": pmean, "gap": gap}
    with open(args.out, "w") as fh:
        json.dump(results, fh, indent=2)
    print(f"\nWritten to {args.out}")
    print("\nRerun with --labels labels_id_matched.json before drawing any")
    print("conclusion. A gain that appears only in the full cohort is not a gain.")

In [6]:
import sys
sys.argv = ["09_improved_features.py"] + ARGS
main()

Patients: 172  (86 pos / 86 neg)
Normalisation: PQN

  profile_1d       (172, 80000)
  map_2d_flat      (172, 32768)
  rt_profile       (172, 64)
  map_2d_rt8       (172, 4096)

dev 137 | test 35

features         model                   CV AUC     TEST
------------------------------------------------------------
profile_1d       plsda_3       0.5532 +/- 0.1002   0.5458
profile_1d       plsda_5       0.5557 +/- 0.0922   0.6536
profile_1d       plsda_10      0.5497 +/- 0.0987   0.6405
profile_1d       ridge_C0.01   0.5693 +/- 0.1069   0.6242
profile_1d       ridge_C0.1    0.5604 +/- 0.1001   0.6307

map_2d_flat      plsda_3       0.5022 +/- 0.0963   0.4706
map_2d_flat      plsda_5       0.5457 +/- 0.1107   0.4739
map_2d_flat      plsda_10      0.5419 +/- 0.1211   0.4444
map_2d_flat      ridge_C0.01   0.5448 +/- 0.1059   0.4935
map_2d_flat      ridge_C0.1    0.5541 +/- 0.1089   0.4902

rt_profile       plsda_3       0.5965 +/- 0.0801   0.5163
rt_profile       plsda_5       0.5826 +/- 0.0